# 가중치 민감도 검증 — 30/30/20/20을 흔들면 판정이 얼마나 바뀌나

산식 가중치(주행거리 30 / 생활권 안 안전 30 / 밖 안전 20 / 패턴 안정성 20)를
바꿔가며 180개 시나리오의 연간 우대 판정이 얼마나 달라지는지 본다.
5단위·합 100·각 축 최소 5인 **969개 조합 전수**를 돌리고, 4축 각각을 같은
조건에서 분해한다. 판정 규칙(75점·12개월 중 9개월·커버리지 80%·미관측 축
재정규화)은 엔진과 동일하게 재현했다.


In [ ]:
# 데이터 로드 — 로컬 저장소가 있으면 그 파일을, 없으면(Colab) GitHub에서 받는다
import json, os, urllib.request
from pathlib import Path

LOCAL = Path("data/fixtures/gaip_simulation_bundle.json")
RAW_URL = ("https://raw.githubusercontent.com/summit1123/seniorcareservice/"
           "claude/gaip-dashboard-refine/data/fixtures/gaip_simulation_bundle.json")

if LOCAL.exists():
    bundle = json.loads(LOCAL.read_text(encoding="utf-8"))
elif Path("../data/fixtures/gaip_simulation_bundle.json").exists():
    bundle = json.loads(Path("../data/fixtures/gaip_simulation_bundle.json").read_text(encoding="utf-8"))
else:
    print("로컬 파일이 없어 GitHub에서 내려받습니다...")
    with urllib.request.urlopen(RAW_URL) as r:
        bundle = json.loads(r.read().decode("utf-8"))

drivers = bundle["drivers"]
print(f"시나리오 {len(drivers)}건 로드 완료")

In [ ]:
# 재판정 함수 — 엔진의 우대 규칙을 그대로 재현
AXES = ("mileage_score", "in_zone_safe_score", "out_zone_safe_score", "pattern_stability_score")
REWARD_THRESHOLD = 75.0
REWARD_REQUIRED_MONTHS = 9
MIN_COVERAGE_PCT = 80.0

def annual_reward_states(drivers, weights):
    w = dict(zip(AXES, weights))
    states = []
    for d in drivers:
        reward_months = eligible_months = 0
        for m in d["monthly_results"]:
            if m["period_role"] != "evaluation":
                continue
            if not m.get("zone_available") or float(m.get("data_coverage_pct", 0)) < MIN_COVERAGE_PCT:
                continue  # 보류 — 판정 미진입
            observed = [(float(m[a]), w[a]) for a in AXES if m.get(a) is not None]
            ow = sum(wt for _, wt in observed)
            if ow <= 0:
                continue
            eligible_months += 1
            score = sum(v * wt for v, wt in observed) / ow  # 미관측 축 재정규화
            if score >= REWARD_THRESHOLD:
                reward_months += 1
        if eligible_months < REWARD_REQUIRED_MONTHS:
            states.append("hold")
        elif reward_months >= REWARD_REQUIRED_MONTHS:
            states.append("reward")
        else:
            states.append("neutral")
    return states

In [ ]:
# 가중치 변형별 재판정
VARIANTS = {
    "30/30/20/20 (현행)": (30, 30, 20, 20),
    "30/30/15/25":        (30, 30, 15, 25),
    "30/30/25/15":        (30, 30, 25, 15),
    "25/35/20/20":        (25, 35, 20, 20),
    "25/25/25/25 (균등)": (25, 25, 25, 25),
    "35/25/20/20":        (35, 25, 20, 20),
    "20/40/20/20":        (20, 40, 20, 20),
    "40/30/15/15":        (40, 30, 15, 15),
}

base = annual_reward_states(drivers, VARIANTS["30/30/20/20 (현행)"])
print(f"현행: 우대 {base.count('reward')} / 중립 {base.count('neutral')} / 보류 {base.count('hold')}\n")

results = {}
print(f"{'가중치':22s} {'우대':>4s} {'현행 대비 변경':>10s}")
print("-" * 44)
for name, weights in VARIANTS.items():
    states = annual_reward_states(drivers, weights)
    changed = sum(1 for a, b in zip(base, states) if a != b)
    results[name] = changed
    print(f"{name:22s} {states.count('reward'):4d} {changed:9d}건")

In [ ]:
# 전체 격자 스윕 — 손으로 고른 변형이 아니라 가중치 공간 전체를 훑는다
import itertools, statistics as st
from collections import defaultdict

grid = [c for c in itertools.product(range(5, 90, 5), repeat=4) if sum(c) == 100]
changes = {}
for w in grid:
    s = annual_reward_states(drivers, w)
    changes[w] = sum(1 for a, b in zip(base, s) if a != b)

vals = list(changes.values())
print(f"전체 격자 {len(grid)}개 조합 — 변경 중앙값 {st.median(vals):.0f}건 · 최대 {max(vals)}건")

near = [w for w in grid if all(abs(a-b) <= 5 for a, b in zip(w, (30,30,20,20)))]
nv = [changes[w] for w in near]
print(f"현행 ±5 이웃 {len(near)}개 — 중앙값 {st.median(nv):.0f}건 · 최대 {max(nv)}건")

# 4축 각각의 효과를 같은 조건에서 본다.
# 한 축을 볼 때 나머지 지배 축(주행거리≤30, 안 안전≥25)은 안정 구간에 고정해
# 교란을 제거한다 — 고정하지 않으면 다른 축의 폭주가 그 축의 효과로 잘못 읽힌다.
AXIS_VIEWS = [
    ("주행거리",  0, lambda w: w[1] >= 25,                  "안 안전≥25 고정"),
    ("안 안전",   1, lambda w: w[0] <= 30,                  "주행거리≤30 고정"),
    ("밖 안전",   2, lambda w: w[0] <= 30 and w[1] >= 25,   "주행거리≤30·안 안전≥25 고정"),
    ("패턴 안정성", 3, lambda w: w[0] <= 30 and w[1] >= 25, "주행거리≤30·안 안전≥25 고정"),
]

profiles = {}
for name, idx, keep, note in AXIS_VIEWS:
    d = defaultdict(list)
    for w, c in changes.items():
        if keep(w):
            d[w[idx]].append(c)
    profiles[name] = {k: st.mean(v) for k, v in sorted(d.items()) if k <= 50 and len(v) >= 3}
    row = "  ".join(f"{k}:{m:5.1f}" for k, m in profiles[name].items())
    span = max(profiles[name].values()) - min(profiles[name].values())
    print(f"\n■ {name} ({note}) — 진폭 {span:.0f}건")
    print(f"  {row}")

stable = [w for w in grid if w[0] <= 30 and w[1] >= 25]
sv = [changes[w] for w in stable]
print(f"\n두 안정 구간을 지키는 {len(stable)}개 조합 — 중앙값 {st.median(sv):.0f}건 · 최대 {max(sv)}건")


In [ ]:
# 시각화 — 4축 전부, 같은 조건·같은 눈금에서
# Colab에는 한글 폰트가 없어 축 이름이 빈 네모로 나온다 → 나눔폰트 설치, 실패 시 영문 라벨
import matplotlib, matplotlib.pyplot as plt

def korean_font():
    have = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
    for c in ("AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
        if c in have:
            return c
    try:  # Colab
        import subprocess
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"],
                       check=True, capture_output=True)
        matplotlib.font_manager.fontManager.addfont(
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
        return "NanumGothic"
    except Exception:
        return None

font = korean_font()
if font:
    plt.rcParams["font.family"] = font
plt.rcParams["axes.unicode_minus"] = False
ko = font is not None

# 현행 값과 각 패널의 제목(한글/영문)
CURRENT = {"주행거리": 30, "안 안전": 30, "밖 안전": 20, "패턴 안정성": 20}
TITLE_KO = {
    "주행거리":   "주행거리 — 35부터 급증 (지배)",
    "안 안전":    "생활권 안 안전 — 줄이면 급증, 30이 바닥 (지배)",
    "밖 안전":    "생활권 밖 안전 — 거의 평평",
    "패턴 안정성": "패턴 안정성 — 완만",
}
TITLE_EN = {
    "주행거리":   "Mileage - jumps from 35 (dominant)",
    "안 안전":    "In-zone safety - min at 30 (dominant)",
    "밖 안전":    "Out-zone safety - nearly flat",
    "패턴 안정성": "Pattern stability - gentle",
}
XLAB_EN = {"주행거리": "mileage weight", "안 안전": "in-zone weight",
           "밖 안전": "out-zone weight", "패턴 안정성": "pattern weight"}

ymax = max(m for p in profiles.values() for m in p.values()) * 1.15
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharey=True)

for ax, (name, prof) in zip(axes.ravel(), profiles.items()):
    ks = list(prof)
    cur = CURRENT[name]
    ax.bar([str(k) for k in ks], [prof[k] for k in ks],
           color=["#0f766e" if k == cur else "#94a3b8" for k in ks])
    ax.set_title(TITLE_KO[name] if ko else TITLE_EN[name], fontsize=11)
    ax.set_xlabel(f"{name} 가중치 (현행 {cur})" if ko
                  else f"{XLAB_EN[name]} (current {cur})")
    ax.set_ylim(0, ymax)
    span = max(prof.values()) - min(prof.values())
    ax.text(0.97, 0.92, (f"진폭 {span:.0f}건" if ko else f"span {span:.0f}"),
            transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#475569")

for ax in axes[:, 0]:
    ax.set_ylabel("평균 판정 변경 (건/180)" if ko else "avg. verdict changes (of 180)")

fig.suptitle("가중치 축별 민감도 — 나머지 지배 축은 안정 구간에 고정" if ko
             else "Per-axis sensitivity (other dominant axes held in stable range)",
             fontsize=13)
plt.tight_layout()
plt.show()


## 해석

- **현행 값 주변(±5 이웃 19개 전부)에서는 변경 중앙값 4건, 최대 12건.** 정확한 값
  논쟁(30이냐 28이냐)은 결과를 크게 바꾸지 않는다.
- **전체 공간은 민감하다** — 969개 조합에서 중앙값 23건, 최대 88건. "아무 가중치나
  된다"는 주장이 아니다.

**4축을 같은 조건에서 비교하면 지배하는 축은 둘, 완만한 축이 둘이다.**
(한 축을 볼 때 나머지 지배 축은 안정 구간에 고정 — 고정하지 않으면 다른 축의
폭주가 그 축의 효과로 잘못 읽힌다.)

| 축 | 5 | 15 | 25 | 30 | 35 | 45 | 진폭 |
|---|---|---|---|---|---|---|---|
| 주행거리 | 16.3 | 12.9 | 7.9 | **6.9** | 12.4 | 26.9 | **27건** |
| 생활권 안 안전 | 30.3 | 21.0 | 9.2 | **6.8** | 7.9 | 13.2 | **23건** |
| 생활권 밖 안전 | 13.3 | 11.8 | 11.8 | 11.7 | 12.1 | 14.5 | 4건 |
| 패턴 안정성 | 18.3 | 14.4 | 10.4 | 8.4 | 7.3 | 6.5 | 12건 |

1. **주행거리는 키우면 안 된다(≤30).** 30 이하에서는 평평하다가 35부터 급증한다.
   적게 몰수록 유리한 축이라, 키우면 활동적인 안전 운전자가 우대에서 탈락한다.
2. **안 안전은 줄이면 안 된다(≥25).** U자를 그리며 **현행 30이 바닥이다**(6.8건).
   5로 줄이면 30건까지 치솟는다. 행동 증거가 가장 많이 쌓이는 축이라, 줄이면
   판정의 기반이 흔들린다.
3. **밖 안전은 어디에 둬도 결과가 거의 같다**(진폭 4건). 생활권 밖 주행이 판정을
   좌우한다는 오해와 반대되는 실측이다.
4. **패턴 안정성은 완만한 내리막**(진폭 12건). 키우면 조금 더 조용해지지만,
   그만큼 다른 축의 몫을 가져가므로 20에서 멈췄다.

- 두 안정 구간(주행거리≤30, 안 안전≥25)을 지키는 371개 조합에서는 변경이
  중앙값 13건, 최대 24건이다. 현행 30/30/20/20은 그 안에 있다.
- **한계**: 합성 시나리오 기반이므로 "이 값이 옳다"의 증거가 아니다. 실측 손해율이
  쌓이면 재보정한다(가중치는 전부 선언적 설정값).
